In [3]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "learning"

In [4]:
from langchain_openai import ChatOpenAI
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
load_dotenv()

creative_llm = ChatOpenAI(
    model="gpt-5-mini",
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
    temperature=0.9
)

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
    temperature=0.0
)
llm1 = ChatOpenRouter(
    model = "openai/gpt-oss-20b"
)
llm2 = ChatOpenRouter(
    model = "nvidia/nemotron-nano-9b-v2"
)
# print(llm.invoke("What is the capital of India?").content)
# print(llm1.invoke("What is the capital of India?").content)

/home/harsh.dhandha@simform.dom/hh/GenAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
print(llm2.invoke("What can u do ? explain transforeners in one line").content)

I can answer questions, explain concepts, assist with writing, and more. Transformers are neural network architectures that use self-attention mechanisms to process sequential data efficiently, enabling tasks like language translation and text generation.



### Tools

Tools are a way augment our LLMs with code execution. A tool is simply a function formatted so that our agent can undertstand how to use it, and then execute it. Let's start by creating a few simple tools.

We can use the @tool decorator to create an LLM-compatible tool from a standard python function — this function should include a few things for optimal performance:

    - A docstring describing what the tool does and when it should be used, this will be read by our LLM/agent and used to decide when to use the tool, and also how to use the tool.

    - Clear parameter names that ideally tell the LLM what each parameter is, if it isn't clear we make sure the docstring explains what the parameter is for and how to use it.

    - Both parameter and return type annotations.


In [6]:
from langchain_core.tools import tool
@tool
def add(x:float, y:float) -> float:
    """ Add 'x' and 'y' ."""
    return x+y
@tool
def multiply(x:float, y:float) -> float:
    """ Multiply 'x' and 'y' ."""
    return x*y
@tool
def exponentiate(x:float, y:float) -> float:
    """ Raise 'x' to the power of 'y' ."""
    return x**y
@tool
def subtract(x:float, y:float) -> float:
    """ Subtract 'x' from 'y' ."""
    return y-x



In [7]:
add

StructuredTool(name='add', description="Add 'x' and 'y' .", args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x716d97170c20>)

In [8]:
print(f"{add.name=}\n{add.description=}")

add.name='add'
add.description="Add 'x' and 'y' ."


In [9]:
add.args_schema.model_json_schema()

{'description': "Add 'x' and 'y' .",
 'properties': {'x': {'title': 'X', 'type': 'number'},
  'y': {'title': 'Y', 'type': 'number'}},
 'required': ['x', 'y'],
 'title': 'add',
 'type': 'object'}

In [10]:
exponentiate.args_schema.model_json_schema()

{'description': "Raise 'x' to the power of 'y' .",
 'properties': {'x': {'title': 'X', 'type': 'number'},
  'y': {'title': 'Y', 'type': 'number'}},
 'required': ['x', 'y'],
 'title': 'exponentiate',
 'type': 'object'}

When invoking the tool, a JSON string output by the LLM will be parsed into JSON and then consumed as kwargs, similar to the below:

In [11]:
import json

llm_output_string = "{\"x\": 5, \"y\": 2}"  # this is the output from the LLM
llm_output_dict = json.loads(llm_output_string)  # load as dictionary
llm_output_dict

{'x': 5, 'y': 2}

This is then passed into the tool function as kwargs (keyword arguments) as indicated by the ** operator - the ** operator is used to unpack the dictionary into keyword arguments.

In [12]:
exponentiate.func(**llm_output_dict)

25

### Creating an Agent

We're going to construct a simple tool calling agent. We will use LangChain Epression Language (LCEL) to construct the agent. We will cover LCEL more in the next chapter, but for now - all we need to know is that our agent will be constructed using syntax and components like so:

agent = (
    <input parameters, including chat history and user query>
    | prompt
    | LLM with tools
)

We need this agent to remember previous interactions within the conversation. To do that, we will use the ChatPromptTemplate with a system message, a placeholder for our chat history, a placeholder for the user query, and finally a placeholder for the agent scratchpad.

The agent scratchpad is where the agent will write it's "notes" as it is working through multiple internal thought and tool-use steps to produce a final output to the user.

In [20]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system" , "you are an helpful assistant"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

In [21]:
from langchain_classic.memory import ConversationBufferMemory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)


In [22]:
from langchain_classic.agents import create_tool_calling_agent

tools = [add, subtract, multiply, exponentiate]

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [23]:
agent.invoke({
    "input": "what is 10.7 multiplied by 7.68?",
    "chat_history": memory.chat_memory.messages,
    "intermediate_steps": []  # agent will append it's internal steps here
})

[ToolAgentAction(tool='multiply', tool_input={'x': 10.7, 'y': 7.68}, log="\nInvoking: `multiply` with `{'x': 10.7, 'y': 7.68}`\n\n\n", message_log=[AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 151, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DgoSze5g5rDNa2l9GQcA9Rkw68wYr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3a64-b1ef-7c30-b043-fb91a880e727-0', tool_calls=[{'name': 'multiply', 'args': {'x': 10.7, 'y': 7.68}, 'id': 'call_MyPKsdjdCVOLrV8q8NT3IOU7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 151, 'outpu

In [24]:
from langchain_classic.agents import AgentExecutor

agent_exe=AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True
)

In [25]:
agent_exe.invoke({
    "input": "what is 10.7 multiplied by 7.68?",
    "chat_history": memory.chat_memory.messages,   
})



> Entering new AgentExecutor chain...

Invoking: `multiply` with `{'x': 10.7, 'y': 7.68}`


82.1759999999999910.7 multiplied by 7.68 is approximately 82.18.

> Finished chain.


{'input': 'what is 10.7 multiplied by 7.68?',
 'chat_history': [HumanMessage(content='what is 10.7 multiplied by 7.68?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='10.7 multiplied by 7.68 is approximately 82.18.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'output': '10.7 multiplied by 7.68 is approximately 82.18.'}

In [27]:
10.7*7.68

82.17599999999999

In [28]:
agent_exe.invoke({
    "input": "My name is Harsh",
    "chat_history": memory
})



> Entering new AgentExecutor chain...
Nice to meet you, Harsh! How can I assist you today?

> Finished chain.


{'input': 'My name is Harsh',
 'chat_history': [HumanMessage(content='what is 10.7 multiplied by 7.68?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='10.7 multiplied by 7.68 is approximately 82.18.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='My name is Harsh', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Nice to meet you, Harsh! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'output': 'Nice to meet you, Harsh! How can I assist you today?'}

In [29]:
agent_exe.invoke({
    "input": "What is nine plus 10, minus 4 * 2, to the power of 3",
    "chat_history": memory
})



> Entering new AgentExecutor chain...

Invoking: `add` with `{'x': 9, 'y': 10}`


19.0
Invoking: `multiply` with `{'x': 4, 'y': 2}`


8.0
Invoking: `exponentiate` with `{'x': 2, 'y': 3}`


8.0
Invoking: `subtract` with `{'x': 19, 'y': 8}`


-11.0The result of \( 9 + 10 - 4 \times 2^3 \) is \(-11\).

> Finished chain.


{'input': 'What is nine plus 10, minus 4 * 2, to the power of 3',
 'chat_history': [HumanMessage(content='what is 10.7 multiplied by 7.68?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='10.7 multiplied by 7.68 is approximately 82.18.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='My name is Harsh', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Nice to meet you, Harsh! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is nine plus 10, minus 4 * 2, to the power of 3', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The result of \\( 9 + 10 - 4 \\times 2^3 \\) is \\(-11\\).', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'output': 'The result of \\( 9 + 10 - 4 \\times 2^3 \\) is \\(-11\\).'}

In [30]:
9+10-(4*2)**3


-493

In [31]:
agent_exe.invoke({
    "input": "What is my name",
    "chat_history": memory
})



> Entering new AgentExecutor chain...
Your name is Harsh.

> Finished chain.


{'input': 'What is my name',
 'chat_history': [HumanMessage(content='what is 10.7 multiplied by 7.68?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='10.7 multiplied by 7.68 is approximately 82.18.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='My name is Harsh', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Nice to meet you, Harsh! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is nine plus 10, minus 4 * 2, to the power of 3', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The result of \\( 9 + 10 - 4 \\times 2^3 \\) is \\(-11\\).', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='What is my name', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Your name is Harsh.', additional_kwargs={}, response_

In [ ]:


os.environ["SERPAPI_API_KEY"] = os.getenv("SERPAPI_API_KEY") \
    or getpass("Enter your SerpAPI API key: ")



In [ ]:


from langchain.agents import load_tools

toolbox = load_tools(tool_names=['serpapi'], llm=llm)



In [ ]:
import requests
from datetime import datetime

@tool
def get_location_from_ip():
    """Get the geographical location based on the IP address."""
    try:
        response = requests.get("https://ipinfo.io/json")
        data = response.json()
        if 'loc' in data:
            latitude, longitude = data['loc'].split(',')
            data = (
                f"Latitude: {latitude},\n"
                f"Longitude: {longitude},\n"
                f"City: {data.get('city', 'N/A')},\n"
                f"Country: {data.get('country', 'N/A')}"
            )
            return data
        else:
            return "Location could not be determined."
    except Exception as e:
        return f"Error occurred: {e}"

@tool
def get_current_datetime() -> str:
    """Return the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [ ]:


tools = toolbox + [get_current_datetime, get_location_from_ip]

agent = create_tool_calling_agent(
    llm=llm, tools=tools, prompt=prompt
)

agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True
)



In [ ]:


out = agent_executor.invoke({
    "input": (
        "I have a few questions, what is the date and time right now? "
        "How is the weather where I am? Please give me degrees in Celsius"
    )
})



In [ ]:
from IPython.display import display, Markdown

display(Markdown(out["output"]))